# MIRAGE++ vs Classical Baselines: Comprehensive Comparison

Compares MIRAGE++ (mirror-descent, KL-entropy) against OLS, Ridge, Lasso, and ElasticNet
across six quantitative-finance regression tasks.

**Sections**
1. Alpha Signal Combination
2. Sparse Portfolio Allocation
3. Volatility Forecasting
4. Factor Exposure Estimation
5. Ensemble Forecasting
6. Macro Predictive Regression
7. Option Surface Fitting
8. Large-Scale Stress Tests
9. Summary & Ranking


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

from mirror_linear_regression import MirrorLinearRegression
from mirror_linear_regression.utils_math import entropy, herfindahl_index, effective_number_of_bets
from examples.synthetic_datasets import (
    toy_alpha_signal_combination,
    toy_sparse_portfolio,
    toy_volatility_forecasting,
    toy_factor_exposures,
    toy_ensemble_forecasting,
    toy_option_surface,
    toy_macro_predictive,
    large_alpha_signals,
    large_portfolio,
    correlated_signals,
    heavy_tail_returns,
    sparse_true_weights,
    adversarial_collinear,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")


In [ ]:
import copy, time

def _simplex(coef):
    c = np.clip(np.asarray(coef, float).flatten(), 0, None)
    s = c.sum()
    return c / s if s > 1e-12 else np.ones(len(c)) / len(c)

def _cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 0.0

def _mirage(lam=0.01, lr=0.10, iters=400, opt="mirror_descent"):
    return MirrorLinearRegression(optimizer=opt, lam=lam,
                                  learning_rate=lr, n_iters=iters, tol=1e-7)

def eval_weights(theta):
    return {"H": float(entropy(theta)),
            "HHI": float(herfindahl_index(theta)),
            "ENB": float(effective_number_of_bets(theta))}

def run_cv(models, X, y, n_splits=5, seed=42):
    # k-fold CV; returns {name: {mse, r2, time, theta}}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    results = {name: {"mse": [], "r2": [], "time": [], "theta": None}
               for name in models}
    for tr, te in kf.split(X):
        Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
        for name, (mtype, mobj) in models.items():
            m = copy.deepcopy(mobj)
            t0 = time.perf_counter()
            m.fit(Xtr, ytr)
            elapsed = time.perf_counter() - t0
            if mtype == "sk":
                theta = _simplex(m.coef_)
                yhat_te = Xte @ theta
            else:
                theta = m.weights
                yhat_te = m.predict(Xte)
            results[name]["mse"].append(mean_squared_error(yte, yhat_te))
            results[name]["r2"].append(r2_score(yte, yhat_te))
            results[name]["time"].append(elapsed)
            results[name]["theta"] = theta
    for r in results.values():
        r["mse_mean"]  = float(np.mean(r["mse"]))
        r["mse_std"]   = float(np.std(r["mse"]))
        r["r2_mean"]   = float(np.mean(r["r2"]))
        r["time_mean"] = float(np.mean(r["time"]))
    return results

def print_results(results, true_w=None):
    hdr = f"{'Model':<28} {'MSE':>10} {'+-':>8} {'R2':>8} {'Time(s)':>9}"
    if true_w is not None:
        hdr += f" {'Cosine':>8}"
    print(hdr)
    print("-" * len(hdr))
    for name, r in results.items():
        row = (f"{name:<28} {r['mse_mean']:>10.6f} {r['mse_std']:>8.6f}"
               f" {r['r2_mean']:>8.4f} {r['time_mean']:>9.4f}")
        if true_w is not None:
            row += f" {_cosine(r['theta'], true_w):>8.4f}"
        print(row)

def plot_weights(results, title, top_n=20, true_w=None):
    fig, axes = plt.subplots(1, len(results), figsize=(3.5 * len(results), 3.5))
    if len(results) == 1:
        axes = [axes]
    for ax, (name, r) in zip(axes, results.items()):
        theta = r["theta"]
        n = min(top_n, len(theta))
        ax.bar(range(n), theta[:n], color="steelblue", alpha=0.8)
        if true_w is not None:
            ax.bar(range(n), true_w[:n], color="none",
                   edgecolor="red", linewidth=1.5, label="true")
            ax.legend(fontsize=8)
        ent = entropy(theta)
        enb = effective_number_of_bets(theta)
        ax.set_title(f"{name}\nH={ent:.3f} ENB={enb:.1f}", fontsize=9)
        ax.set_xlabel("component")
        ax.set_ylabel("weight")
    fig.suptitle(title, fontweight="bold")
    plt.tight_layout()
    plt.show()

BASE_MODELS = {
    "OLS":            ("sk", LinearRegression(fit_intercept=False)),
    "Ridge(0.01)":    ("sk", Ridge(alpha=0.01,  fit_intercept=False)),
    "Ridge(1.0)":     ("sk", Ridge(alpha=1.0,   fit_intercept=False)),
    "Lasso(0.001)":   ("sk", Lasso(alpha=0.001, fit_intercept=False, max_iter=5000)),
    "ElasticNet":     ("sk", ElasticNet(alpha=0.01, l1_ratio=0.5,
                                         fit_intercept=False, max_iter=5000)),
    "MIRAGE-MD":      ("mlr", _mirage(0.01)),
    "MIRAGE-NGD":     ("mlr", _mirage(0.01, lr=0.05, opt="natural_gradient")),
    "MIRAGE-MP":      ("mlr", _mirage(0.01, opt="mirror_prox")),
    "MIRAGE-Ada":     ("mlr", _mirage(0.01, lr=0.20, opt="ada_mirror")),
}
print("Helpers defined.")


## 1. Alpha Signal Combination

Known simplex ground truth; tests weight recovery and out-of-sample MSE.


In [ ]:
print("=" * 60)
print("Section 1: Alpha Signal Combination (n=300, m=10)")
print("=" * 60)

X, y, w_true = toy_alpha_signal_combination(n_samples=300, n_signals=10, seed=42)
res1 = run_cv(BASE_MODELS, X, y)
print_results(res1, true_w=w_true)
plot_weights(
    {k: v for k, v in res1.items() if k in ("OLS", "Ridge(1.0)", "MIRAGE-MD")},
    "Alpha Signal Combination -- learned weights (first 10)",
    top_n=10, true_w=w_true
)

m_conv = _mirage(0.01)
m_conv.fit(X, y)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
a1.semilogy(m_conv.loss_history, color="navy")
a1.set_title("Loss convergence"); a1.set_xlabel("iteration"); a1.set_ylabel("loss")
a2.plot(m_conv.entropy_history, color="darkorange")
a2.set_title("Weight entropy H(theta)"); a2.set_xlabel("iteration"); a2.set_ylabel("H")
plt.suptitle("MIRAGE-MD on Alpha Signals", fontweight="bold")
plt.tight_layout(); plt.show()


## 2. Sparse Portfolio Allocation

No ground-truth weights; focus on diversification metrics (HHI, ENB).


In [ ]:
print("=" * 60)
print("Section 2: Portfolio Allocation (T=200, N=20)")
print("=" * 60)

ret = toy_sparse_portfolio(T=200, N=20, seed=1)
X_p, y_p = ret, ret.mean(axis=1)
res2 = run_cv(BASE_MODELS, X_p, y_p)
print_results(res2)

print("\nDiversity metrics on learned portfolio weights:")
print(f"{'Model':<28} {'Entropy':>9} {'HHI':>8} {'ENB':>8}")
print("-" * 56)
for name, r in res2.items():
    d = eval_weights(r["theta"])
    print(f"{name:<28} {d['H']:>9.4f} {d['HHI']:>8.4f} {d['ENB']:>8.2f}")

plot_weights(
    {k: v for k, v in res2.items() if k in ("OLS", "MIRAGE-MD", "MIRAGE-Ada")},
    "Portfolio weights comparison (N=20 assets)"
)


## 3. Volatility Forecasting

Positive coefficients; lambda sensitivity analysis.


In [ ]:
print("=" * 60)
print("Section 3: Volatility Forecasting (T=300, m=8)")
print("=" * 60)

X_v, y_v, w_v = toy_volatility_forecasting(T=300, n_features=8, seed=7)
res3 = run_cv(BASE_MODELS, X_v, y_v)
print_results(res3, true_w=w_v)

lambdas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2]
lam_mse, lam_ent = [], []
kf3 = KFold(n_splits=5, shuffle=True, random_state=42)
for lam in lambdas:
    fold_mse = []
    for tr, te in kf3.split(X_v):
        m = MirrorLinearRegression(lam=lam, n_iters=400, tol=1e-7)
        m.fit(X_v[tr], y_v[tr])
        fold_mse.append(mean_squared_error(y_v[te], m.predict(X_v[te])))
    m_full = MirrorLinearRegression(lam=lam, n_iters=400, tol=1e-7)
    m_full.fit(X_v, y_v)
    lam_mse.append(np.mean(fold_mse))
    lam_ent.append(entropy(m_full.weights))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.5))
a1.semilogx(lambdas, lam_mse, "o-", color="navy")
a1.set_xlabel("lambda"); a1.set_ylabel("CV MSE"); a1.set_title("lambda vs Out-of-sample MSE")
a2.semilogx(lambdas, lam_ent, "s-", color="darkorange")
a2.set_xlabel("lambda"); a2.set_ylabel("H(theta)"); a2.set_title("lambda vs Weight Entropy")
plt.suptitle("Volatility Forecasting -- lambda sensitivity", fontweight="bold")
plt.tight_layout(); plt.show()


## 4. Factor Exposure Estimation

Per-stock weight recovery across 50 stocks, 8 factors.


In [ ]:
print("=" * 60)
print("Section 4: Factor Exposure Estimation (50 stocks, 8 factors)")
print("=" * 60)

F, exposures = toy_factor_exposures(n_stocks=50, n_factors=8, seed=11)
ols_errors, mlr_errors = [], []
ols_cosines, mlr_cosines = [], []

for i in range(len(F)):
    true_w = exposures[i]
    X_s = F
    y_s = F @ true_w + 0.05 * np.random.RandomState(i).randn(len(F))

    ols = LinearRegression(fit_intercept=False).fit(X_s, y_s)
    t_ols = _simplex(ols.coef_)
    ols_errors.append(mean_squared_error(y_s, X_s @ t_ols))
    ols_cosines.append(_cosine(t_ols, true_w))

    mlr = _mirage(0.01)
    mlr.fit(X_s, y_s)
    t_mlr = mlr.weights
    mlr_errors.append(mean_squared_error(y_s, mlr.predict(X_s)))
    mlr_cosines.append(_cosine(t_mlr, true_w))

print(f"{'Method':<15} {'Mean MSE':>12} {'Mean Cosine':>13}")
print("-" * 42)
print(f"{'OLS':<15} {np.mean(ols_errors):>12.6f} {np.mean(ols_cosines):>13.4f}")
print(f"{'MIRAGE-MD':<15} {np.mean(mlr_errors):>12.6f} {np.mean(mlr_cosines):>13.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(ols_errors, mlr_errors, alpha=0.5, s=20)
mn = min(min(ols_errors), min(mlr_errors))
mx = max(max(ols_errors), max(mlr_errors))
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1)
axes[0].set_xlabel("OLS MSE"); axes[0].set_ylabel("MIRAGE MSE")
axes[0].set_title("Per-stock MSE: OLS vs MIRAGE")
axes[1].hist(mlr_cosines, bins=20, alpha=0.7, label="MIRAGE", color="steelblue")
axes[1].hist(ols_cosines, bins=20, alpha=0.5, label="OLS", color="orange")
axes[1].set_xlabel("cosine similarity to true exposure")
axes[1].set_title("Weight recovery distribution"); axes[1].legend()
plt.tight_layout(); plt.show()


## 5. Ensemble Forecasting

Model combination with 15 signals; entropy regularization encourages diversification.


In [ ]:
print("=" * 60)
print("Section 5: Ensemble Forecasting (n=300, 15 models)")
print("=" * 60)

X_e, y_e, w_e = toy_ensemble_forecasting(n_models=15, n_obs=300, seed=21)
res5 = run_cv(BASE_MODELS, X_e, y_e)
print_results(res5, true_w=w_e)

print("\nEnsemble weight diversification:")
print(f"{'Model':<28} {'Cosine':>8} {'ENB':>8} {'HHI':>8}")
print("-" * 56)
for name, r in res5.items():
    cos = _cosine(r["theta"], w_e)
    d = eval_weights(r["theta"])
    print(f"{name:<28} {cos:>8.4f} {d['ENB']:>8.2f} {d['HHI']:>8.4f}")

plot_weights(
    {k: v for k, v in res5.items() if k in ("OLS", "Lasso(0.001)", "MIRAGE-MD")},
    "Ensemble model weights (15 models)", top_n=15, true_w=w_e
)


## 6. Macro Predictive Regression

Ground truth has negative weights -- simplex projection is an intentional constraint.


In [ ]:
print("=" * 60)
print("Section 6: Macro Predictive (n=300, 12 features)")
print("=" * 60)

X_m, y_m, _ = toy_macro_predictive(n_obs=300, n_features=12, seed=31)
res6 = run_cv(BASE_MODELS, X_m, y_m)
print_results(res6)

families = {
    "sklearn": ["OLS", "Ridge(0.01)", "Ridge(1.0)", "Lasso(0.001)", "ElasticNet"],
    "MIRAGE":  ["MIRAGE-MD", "MIRAGE-NGD", "MIRAGE-MP", "MIRAGE-Ada"],
}
fig, ax = plt.subplots(figsize=(9, 4))
colors = {"sklearn": "steelblue", "MIRAGE": "darkorange"}
positions, labels = [], []
idx = 0
for family, names in families.items():
    for name in names:
        if name in res6:
            r = res6[name]
            ax.bar(idx, r["mse_mean"], color=colors[family], alpha=0.8,
                   yerr=r["mse_std"], capsize=4)
            positions.append(idx); labels.append(name)
            idx += 1
    idx += 0.5
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("CV MSE"); ax.set_title("Macro Predictive -- Out-of-sample MSE")
for family, color in colors.items():
    ax.bar(0, 0, color=color, label=family)
ax.legend()
plt.tight_layout(); plt.show()


## 7. Option Surface Fitting

Polynomial features over implied-vol surface; assesses fit quality and residuals.


In [ ]:
print("=" * 60)
print("Section 7: Option Implied-Vol Surface (100 points)")
print("=" * 60)

strikes, maturities, observed, true_surface = toy_option_surface(n_points=100, seed=17)
X_o = np.column_stack([
    strikes, maturities,
    strikes**2, maturities**2,
    strikes * maturities,
    np.ones(len(strikes)),
])
y_o = observed

models_o = {
    "OLS":         ("sk",  LinearRegression(fit_intercept=False)),
    "Ridge(0.01)": ("sk",  Ridge(alpha=0.01, fit_intercept=False)),
    "MIRAGE-MD":   ("mlr", _mirage(0.01)),
}
res7 = run_cv(models_o, X_o, y_o)
print_results(res7)

m_mirage = _mirage(0.01); m_mirage.fit(X_o, y_o)
m_ols = LinearRegression(fit_intercept=False); m_ols.fit(X_o, y_o)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sc = axes[0].scatter(strikes, maturities, c=true_surface, cmap="viridis", s=30)
plt.colorbar(sc, ax=axes[0]); axes[0].set_title("True implied vol surface")
axes[0].set_xlabel("strike"); axes[0].set_ylabel("maturity")

pred_mlr = m_mirage.predict(X_o)
residuals_mlr = observed - pred_mlr
residuals_ols = observed - (X_o @ _simplex(m_ols.coef_))
axes[1].plot(strikes, residuals_ols, "o", ms=4, alpha=0.5, label="OLS residuals")
axes[1].plot(strikes, residuals_mlr, "s", ms=4, alpha=0.5, label="MIRAGE residuals")
axes[1].axhline(0, color="black", lw=0.8, ls="--")
axes[1].set_xlabel("strike"); axes[1].set_ylabel("residual")
axes[1].set_title("Fit residuals vs strike"); axes[1].legend()
plt.tight_layout(); plt.show()


## 8. Large-Scale Stress Tests

High-dimensional, correlated, heavy-tail, and adversarial scenarios.


In [ ]:
print("=" * 60)
print("Section 8: Large-Scale Stress Tests")
print("=" * 60)

from mirror_linear_regression.convergence import (
    kl_regret_bound, euclidean_regret_bound
)

# 8a. Dimensional scaling
dims = [10, 20, 50, 100, 200, 500]
T_reg, G = 400, 1.0
kl_bounds = [kl_regret_bound(T_reg, n, G) for n in dims]
eu_bounds = [euclidean_regret_bound(T_reg, n, G) for n in dims]
ratios    = [e / k for k, e in zip(kl_bounds, eu_bounds)]

print("\nDimensional scaling -- theoretical regret bounds:")
print(f"{'n':>6} {'KL bound':>12} {'Eucl bound':>12} {'Ratio E/K':>12}")
print("-" * 44)
for n, k, e, r in zip(dims, kl_bounds, eu_bounds, ratios):
    print(f"{n:>6} {k:>12.4f} {e:>12.4f} {r:>12.2f}x")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dims, kl_bounds, "o-", label="KL (MIRAGE)", color="navy")
ax.plot(dims, eu_bounds, "s-", label="Euclidean",   color="orange")
ax.set_xlabel("dimension n"); ax.set_ylabel("regret bound")
ax.set_title("KL vs Euclidean regret bound scaling")
ax.legend(); plt.tight_layout(); plt.show()

# 8b. High-dimensional alpha n=500
print("\n8b. High-dimensional alpha signals (m=1000, n=500)...")
X_hd, y_hd, w_hd = large_alpha_signals(n_signals=500, n_samples=1000, seed=100)
t0 = time.perf_counter()
m_hd = MirrorLinearRegression(lam=0.01, n_iters=300, tol=1e-6)
m_hd.fit(X_hd, y_hd)
t_hd = time.perf_counter() - t0
nonzero = int(np.sum(w_hd > 0))
print(f"  Runtime: {t_hd:.2f}s | ENB: {effective_number_of_bets(m_hd.weights):.1f}"
      f" | True non-zeros: {nonzero}")

# 8c. Correlated signals
print("\n8c. Correlated signals (rho=0.95)...")
X_cor, y_cor, w_cor = correlated_signals(n_signals=30, n_obs=300, rho=0.95, seed=300)
m_cor = _mirage(0.01); m_cor.fit(X_cor, y_cor)
ols_cor = LinearRegression(fit_intercept=False).fit(X_cor, y_cor)
mse_mlr = mean_squared_error(y_cor, m_cor.predict(X_cor))
mse_ols = mean_squared_error(y_cor, X_cor @ _simplex(ols_cor.coef_))
print(f"  MIRAGE MSE: {mse_mlr:.6f} | OLS MSE: {mse_ols:.6f}")
print(f"  MIRAGE cosine: {_cosine(m_cor.weights, w_cor):.4f}"
      f" | OLS cosine: {_cosine(_simplex(ols_cor.coef_), w_cor):.4f}")

# 8d. Heavy-tail returns
print("\n8d. Heavy-tail returns (Student-t df=3)...")
ret_ht = heavy_tail_returns(T=500, N=30, df=3, seed=400)
X_ht, y_ht = ret_ht, ret_ht.mean(axis=1)
m_ht = _mirage(0.01); m_ht.fit(X_ht, y_ht)
ols_ht = LinearRegression(fit_intercept=False).fit(X_ht, y_ht)
mse_mlr_ht = mean_squared_error(y_ht, m_ht.predict(X_ht))
mse_ols_ht = mean_squared_error(y_ht, X_ht @ _simplex(ols_ht.coef_))
print(f"  MIRAGE MSE: {mse_mlr_ht:.8f} | OLS MSE: {mse_ols_ht:.8f}")

# 8e. Adversarial collinear
print("\n8e. Adversarial collinear (near-duplicate features)...")
X_adv, y_adv, w_adv = adversarial_collinear(n=30, n_dupes=10, n_obs=300, seed=700)
m_adv = _mirage(0.01)
try:
    m_adv.fit(X_adv, y_adv)
    mse_adv = mean_squared_error(y_adv, m_adv.predict(X_adv))
    enb_adv = effective_number_of_bets(m_adv.weights)
    print(f"  MIRAGE converged: MSE={mse_adv:.6f} ENB={enb_adv:.2f}")
except Exception as ex:
    print(f"  MIRAGE failed: {ex}")


## 9. Summary & Ranking

Win counts, average rank, and diversification metrics across all tasks.


In [ ]:
print("=" * 60)
print("Section 9: Summary & Overall Ranking")
print("=" * 60)

all_results = {
    "Alpha(10)":  res1,
    "Portfolio":  res2,
    "Volatility": res3,
    "Ensemble":   res5,
    "Macro":      res6,
    "OptionSurf": res7,
}

wins = {}
for ds_name, ds_res in all_results.items():
    valid = {m: r["mse_mean"] for m, r in ds_res.items() if "mse_mean" in r}
    best = min(valid.values())
    for m, v in valid.items():
        if abs(v - best) < 1e-9:
            wins[m] = wins.get(m, 0) + 1

n_ds = len(all_results)
print(f"\nWin counts across {n_ds} datasets (out-of-sample MSE):")
print("-" * 50)
for model, w in sorted(wins.items(), key=lambda x: -x[1]):
    bar = "#" * w + "." * (n_ds - w)
    print(f"  {model:<28}  {bar}  {w}/{n_ds}")

avg_ranks = {}
for ds_res in all_results.values():
    sorted_m = sorted([(m, ds_res[m]["mse_mean"]) for m in BASE_MODELS
                       if m in ds_res], key=lambda x: x[1])
    for rank, (m, _) in enumerate(sorted_m, 1):
        avg_ranks.setdefault(m, []).append(rank)

print("\nAverage MSE rank across datasets (lower is better):")
print("-" * 50)
ranked = sorted([(m, np.mean(ranks)) for m, ranks in avg_ranks.items()],
                key=lambda x: x[1])
for m, avg_rank in ranked:
    print(f"  {m:<28}  avg rank = {avg_rank:.2f}")

print("\nDiversification on Ensemble task:")
print(f"  {'Model':<28} {'ENB':>8} {'HHI':>8}")
for name in ["OLS", "Ridge(1.0)", "Lasso(0.001)", "MIRAGE-MD", "MIRAGE-Ada"]:
    if name in res5:
        d = eval_weights(res5[name]["theta"])
        print(f"  {name:<28} {d['ENB']:>8.2f} {d['HHI']:>8.4f}")

print("\n" + "=" * 60)
print("DONE -- MIRAGE++ Comprehensive Comparison Complete")
print("=" * 60)
